In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SparkLocalJupyter")
    .master("local[*]")   # utilise tous les cœurs
    .getOrCreate()
)

spark

## 📘 Exercice 1 — Charger les données

In [5]:
clients = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("./data/clients.csv")

commandes = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("./data/commandes.csv")

clients.show(5)
commandes.show(5)

clients.printSchema()
commandes.printSchema()


+---------+------+----+-----------+---------+-----------+
|client_id|   nom| age|       pays|    ville|    segment|
+---------+------+----+-----------+---------+-----------+
|        1|Durand|  28|     France|    Paris|Particulier|
|        2|Martin|  45|     France|     Lyon| Entreprise|
|        3|Dupont|  33|   Belgique|Bruxelles|Particulier|
|        4|Moreau|NULL|     France|Marseille|Particulier|
|        5| Smith|  52|Royaume-Uni|   London| Entreprise|
+---------+------+----+-----------+---------+-----------+
only showing top 5 rows

+-----------+---------+-------------+-------+--------+
|commande_id|client_id|date_commande|montant|  status|
+-----------+---------+-------------+-------+--------+
|       1001|        1|   2024-01-05|  120.5|  LIVREE|
|       1002|        1|   2024-01-15|   80.0| ANNULEE|
|       1003|        2|   2024-02-01|  560.0|  LIVREE|
|       1004|        3|   2024-02-10|   42.0|  LIVREE|
|       1005|        4|   2024-02-11|   NULL|EN_COURS|
+-----------+

## 📘 Exercice 2 — Sélection et filtrage

In [6]:
clients.select("nom", "age", "pays").show()

clients.filter(clients.age > 30).show()

clients.filter(clients.pays == "France").show()


+------+----+-----------+
|   nom| age|       pays|
+------+----+-----------+
|Durand|  28|     France|
|Martin|  45|     France|
|Dupont|  33|   Belgique|
|Moreau|NULL|     France|
| Smith|  52|Royaume-Uni|
| Rossi|  41|     Italie|
|Garcia|  29|    Espagne|
|Müller|  38|  Allemagne|
| Lopez|  27|    Espagne|
|Nguyen|  31|     France|
+------+----+-----------+

+---------+------+---+-----------+---------+-----------+
|client_id|   nom|age|       pays|    ville|    segment|
+---------+------+---+-----------+---------+-----------+
|        2|Martin| 45|     France|     Lyon| Entreprise|
|        3|Dupont| 33|   Belgique|Bruxelles|Particulier|
|        5| Smith| 52|Royaume-Uni|   London| Entreprise|
|        6| Rossi| 41|     Italie|     Rome|Particulier|
|        8|Müller| 38|  Allemagne|   Berlin| Entreprise|
|       10|Nguyen| 31|     France| Toulouse|Particulier|
+---------+------+---+-----------+---------+-----------+

+---------+------+----+------+---------+-----------+
|client_id|

## 📘 Exercice 3 — Nettoyage des données

In [7]:
clients.select("nom", "age", "pays").show()

clients.filter(clients.age > 30).show()

clients.filter(clients.pays == "France").show()


+------+----+-----------+
|   nom| age|       pays|
+------+----+-----------+
|Durand|  28|     France|
|Martin|  45|     France|
|Dupont|  33|   Belgique|
|Moreau|NULL|     France|
| Smith|  52|Royaume-Uni|
| Rossi|  41|     Italie|
|Garcia|  29|    Espagne|
|Müller|  38|  Allemagne|
| Lopez|  27|    Espagne|
|Nguyen|  31|     France|
+------+----+-----------+

+---------+------+---+-----------+---------+-----------+
|client_id|   nom|age|       pays|    ville|    segment|
+---------+------+---+-----------+---------+-----------+
|        2|Martin| 45|     France|     Lyon| Entreprise|
|        3|Dupont| 33|   Belgique|Bruxelles|Particulier|
|        5| Smith| 52|Royaume-Uni|   London| Entreprise|
|        6| Rossi| 41|     Italie|     Rome|Particulier|
|        8|Müller| 38|  Allemagne|   Berlin| Entreprise|
|       10|Nguyen| 31|     France| Toulouse|Particulier|
+---------+------+---+-----------+---------+-----------+

+---------+------+----+------+---------+-----------+
|client_id|

## 📘 Exercice 4 — Ajout de colonnes

In [10]:
from pyspark.sql.functions import col

c = clients.withColumn("age_plus_1", col("age") + 1)

cmd = commandes.withColumn("montant_TTC", col("montant") * 1.20)

c.show()
cmd.show()

+---------+------+----+-----------+---------+-----------+----------+
|client_id|   nom| age|       pays|    ville|    segment|age_plus_1|
+---------+------+----+-----------+---------+-----------+----------+
|        1|Durand|  28|     France|    Paris|Particulier|        29|
|        2|Martin|  45|     France|     Lyon| Entreprise|        46|
|        3|Dupont|  33|   Belgique|Bruxelles|Particulier|        34|
|        4|Moreau|NULL|     France|Marseille|Particulier|      NULL|
|        5| Smith|  52|Royaume-Uni|   London| Entreprise|        53|
|        6| Rossi|  41|     Italie|     Rome|Particulier|        42|
|        7|Garcia|  29|    Espagne|   Madrid|Particulier|        30|
|        8|Müller|  38|  Allemagne|   Berlin| Entreprise|        39|
|        9| Lopez|  27|    Espagne|Barcelona|Particulier|        28|
|       10|Nguyen|  31|     France| Toulouse|Particulier|        32|
+---------+------+----+-----------+---------+-----------+----------+

+-----------+---------+----------

## 📘 Exercice 5 — Agrégations

In [15]:
from pyspark.sql.functions import avg, sum
clients.groupBy("pays").agg(avg("age")).show()

commandes.groupBy("client_id").agg(sum("montant")).show()

commandes.filter(commandes.status == "LIVREE").count()


+-----------+------------------+
|       pays|          avg(age)|
+-----------+------------------+
|     France|34.666666666666664|
|    Espagne|              28.0|
|  Allemagne|              38.0|
|Royaume-Uni|              52.0|
|     Italie|              41.0|
|   Belgique|              33.0|
+-----------+------------------+

+---------+------------+
|client_id|sum(montant)|
+---------+------------+
|        1|       200.5|
|        6|        75.0|
|        3|        42.0|
|        5|       230.0|
|        4|        NULL|
|        8|       990.0|
|        7|        15.0|
|       10|       120.0|
|        2|       560.0|
+---------+------------+



8

## 📘 Exercice 6 — Jointure

In [18]:
joint = clients.join(commandes, ["client_id"])

joint.select("nom", "pays", "montant") \
     .filter(joint.status == "LIVREE") \
     .show()



+------+-----------+-------+
|   nom|       pays|montant|
+------+-----------+-------+
|Durand|     France|  120.5|
|Martin|     France|  560.0|
|Dupont|   Belgique|   42.0|
| Smith|Royaume-Uni|  230.0|
| Rossi|     Italie|   75.0|
|Garcia|    Espagne|   15.0|
|Müller|  Allemagne|  990.0|
|Nguyen|     France|  120.0|
+------+-----------+-------+



## 📘 Exercice 7 — SQL avec Spark

In [19]:
clients.createOrReplaceTempView("clients_v")

spark.sql("SELECT * FROM clients_v WHERE age > 40").show()

spark.sql("SELECT pays, COUNT(*) FROM clients_v GROUP BY pays").show()


+---------+------+---+-----------+------+-----------+
|client_id|   nom|age|       pays| ville|    segment|
+---------+------+---+-----------+------+-----------+
|        2|Martin| 45|     France|  Lyon| Entreprise|
|        5| Smith| 52|Royaume-Uni|London| Entreprise|
|        6| Rossi| 41|     Italie|  Rome|Particulier|
+---------+------+---+-----------+------+-----------+

+-----------+--------+
|       pays|count(1)|
+-----------+--------+
|     France|       4|
|    Espagne|       2|
|  Allemagne|       1|
|Royaume-Uni|       1|
|     Italie|       1|
|   Belgique|       1|
+-----------+--------+



## 📘 Exercice 8 — UDF (User Defined Function)

In [20]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

upper_udf = udf(lambda s: s.upper(), StringType())

clients.withColumn("nom_upper", upper_udf("nom")).show()


+---------+------+----+-----------+---------+-----------+---------+
|client_id|   nom| age|       pays|    ville|    segment|nom_upper|
+---------+------+----+-----------+---------+-----------+---------+
|        1|Durand|  28|     France|    Paris|Particulier|   DURAND|
|        2|Martin|  45|     France|     Lyon| Entreprise|   MARTIN|
|        3|Dupont|  33|   Belgique|Bruxelles|Particulier|   DUPONT|
|        4|Moreau|NULL|     France|Marseille|Particulier|   MOREAU|
|        5| Smith|  52|Royaume-Uni|   London| Entreprise|    SMITH|
|        6| Rossi|  41|     Italie|     Rome|Particulier|    ROSSI|
|        7|Garcia|  29|    Espagne|   Madrid|Particulier|   GARCIA|
|        8|Müller|  38|  Allemagne|   Berlin| Entreprise|   MÜLLER|
|        9| Lopez|  27|    Espagne|Barcelona|Particulier|    LOPEZ|
|       10|Nguyen|  31|     France| Toulouse|Particulier|   NGUYEN|
+---------+------+----+-----------+---------+-----------+---------+



## 📘 Exercice 9 — Export des résultats

In [22]:
commandes_livrees = commandes.filter(commandes.status == "LIVREE")

commandes_livrees.write.mode("overwrite").csv("output/commandes_livrees.csv")

commandes_livrees.write.parquet("output/commandes_livrees.parquet")


## 📘 Exercice 10 — Pipeline complet

In [23]:
from pyspark.sql.functions import col, sum

c = clients.na.drop()
cmd = commandes.na.fill({"montant": 0})

joint = c.join(cmd, ["client_id"]) \
         .withColumn("montant_TTC", col("montant") * 1.20) \
         .filter(col("status") == "LIVREE")

joint.groupBy("pays").agg(sum("montant_TTC")).show()


+-----------+----------------+
|       pays|sum(montant_TTC)|
+-----------+----------------+
|     France|           960.6|
|    Espagne|            18.0|
|  Allemagne|          1188.0|
|Royaume-Uni|           276.0|
|     Italie|            90.0|
|   Belgique|            50.4|
+-----------+----------------+



# Mini Projet

## Étape 1 – Chargement

In [27]:
from pyspark.sql.functions import col, year, month, sum, count, round, row_number
from pyspark.sql.window import Window

# Étape 1 – Chargement
clients = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("./data/clients.csv")

commandes = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("./data/commandes.csv")

clients.printSchema()
commandes.printSchema()

clients.show(5)
commandes.show(5)

root
 |-- client_id: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- pays: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- segment: string (nullable = true)

root
 |-- commande_id: integer (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- date_commande: date (nullable = true)
 |-- montant: double (nullable = true)
 |-- status: string (nullable = true)

+---------+------+----+-----------+---------+-----------+
|client_id|   nom| age|       pays|    ville|    segment|
+---------+------+----+-----------+---------+-----------+
|        1|Durand|  28|     France|    Paris|Particulier|
|        2|Martin|  45|     France|     Lyon| Entreprise|
|        3|Dupont|  33|   Belgique|Bruxelles|Particulier|
|        4|Moreau|NULL|     France|Marseille|Particulier|
|        5| Smith|  52|Royaume-Uni|   London| Entreprise|
+---------+------+----+-----------+---------+-----------+
only showing top 5 rows

+---------

## Étape 2 – Nettoyage

In [28]:
# Étape 2 – Nettoyage
clients_before = clients.count()
clients_clean = clients.filter(col("age").isNotNull())
clients_after = clients_clean.count()

print(f"Clients avant nettoyage : {clients_before}, après : {clients_after}")

commandes_before = commandes.count()
commandes_clean = commandes.na.fill({"montant": 0.0})
commandes_after = commandes_clean.count()

print(f"Commandes avant nettoyage : {commandes_before}, après : {commandes_after}")


Clients avant nettoyage : 10, après : 9
Commandes avant nettoyage : 10, après : 10


## Étape 3 – Filtrer les commandes pertinentes

In [29]:
# Étape 3 – Filtrer les commandes pertinentes
commandes_valide = commandes_clean \
    .filter(col("status") == "LIVREE") \
    .filter(col("montant") > 0)

print("Nombre de commandes valides :", commandes_valide.count())

# Étape 4 – Jointure
ventes = clients_clean.join(commandes_valide, ["client_id"], "inner") \
    .select(
        col("client_id"),
        col("nom"),
        col("pays"),
        col("date_commande"),
        col("montant")
    )

ventes.show()


Nombre de commandes valides : 8
+---------+------+-----------+-------------+-------+
|client_id|   nom|       pays|date_commande|montant|
+---------+------+-----------+-------------+-------+
|        1|Durand|     France|   2024-01-05|  120.5|
|        2|Martin|     France|   2024-02-01|  560.0|
|        3|Dupont|   Belgique|   2024-02-10|   42.0|
|        5| Smith|Royaume-Uni|   2024-03-05|  230.0|
|        6| Rossi|     Italie|   2024-03-14|   75.0|
|        7|Garcia|    Espagne|   2024-03-20|   15.0|
|        8|Müller|  Allemagne|   2024-03-22|  990.0|
|       10|Nguyen|     France|   2024-03-25|  120.0|
+---------+------+-----------+-------------+-------+



## Étape 5 – Enrichissement

In [31]:
# Étape 5 – Enrichissement
ventes_enrichies = ventes \
    .withColumn("montant_TTC", col("montant") * 1.20) \
    .withColumn("annee", year(col("date_commande"))) \
    .withColumn("mois", month(col("date_commande")))

ventes_enrichies.show()


+---------+------+-----------+-------------+-------+-----------+-----+----+
|client_id|   nom|       pays|date_commande|montant|montant_TTC|annee|mois|
+---------+------+-----------+-------------+-------+-----------+-----+----+
|        1|Durand|     France|   2024-01-05|  120.5|      144.6| 2024|   1|
|        2|Martin|     France|   2024-02-01|  560.0|      672.0| 2024|   2|
|        3|Dupont|   Belgique|   2024-02-10|   42.0|       50.4| 2024|   2|
|        5| Smith|Royaume-Uni|   2024-03-05|  230.0|      276.0| 2024|   3|
|        6| Rossi|     Italie|   2024-03-14|   75.0|       90.0| 2024|   3|
|        7|Garcia|    Espagne|   2024-03-20|   15.0|       18.0| 2024|   3|
|        8|Müller|  Allemagne|   2024-03-22|  990.0|     1188.0| 2024|   3|
|       10|Nguyen|     France|   2024-03-25|  120.0|      144.0| 2024|   3|
+---------+------+-----------+-------------+-------+-----------+-----+----+



## Étape 6 – KPI

In [32]:
# Étape 6 – KPI

# 6.1 CA par pays
ca_par_pays = ventes_enrichies.groupBy("pays") \
    .agg(round(sum("montant_TTC"), 2).alias("ca_total_TTC"))

ca_par_pays.show()

# 6.2 KPI par client
kpi_clients = ventes_enrichies.groupBy("client_id", "nom", "pays") \
    .agg(
        round(sum("montant_TTC"), 2).alias("ca_client_TTC"),
        count("*").alias("nb_commandes")
    )

kpi_clients.show()


+-----------+------------+
|       pays|ca_total_TTC|
+-----------+------------+
|     France|       960.6|
|    Espagne|        18.0|
|  Allemagne|      1188.0|
|Royaume-Uni|       276.0|
|     Italie|        90.0|
|   Belgique|        50.4|
+-----------+------------+

+---------+------+-----------+-------------+------------+
|client_id|   nom|       pays|ca_client_TTC|nb_commandes|
+---------+------+-----------+-------------+------------+
|        2|Martin|     France|        672.0|           1|
|        5| Smith|Royaume-Uni|        276.0|           1|
|        1|Durand|     France|        144.6|           1|
|       10|Nguyen|     France|        144.0|           1|
|        7|Garcia|    Espagne|         18.0|           1|
|        3|Dupont|   Belgique|         50.4|           1|
|        6| Rossi|     Italie|         90.0|           1|
|        8|Müller|  Allemagne|       1188.0|           1|
+---------+------+-----------+-------------+------------+



## Étape 7 – Top clients par pays (fenêtres)

In [33]:
# Étape 7 – Top clients par pays (fenêtres)
window_pays = Window.partitionBy("pays").orderBy(col("ca_client_TTC").desc())

kpi_clients_ranked = kpi_clients \
    .withColumn("rang_pays", row_number().over(window_pays))

top_clients_par_pays = kpi_clients_ranked \
    .filter(col("rang_pays") <= 3)

top_clients_par_pays.show()


+---------+------+-----------+-------------+------------+---------+
|client_id|   nom|       pays|ca_client_TTC|nb_commandes|rang_pays|
+---------+------+-----------+-------------+------------+---------+
|        8|Müller|  Allemagne|       1188.0|           1|        1|
|        3|Dupont|   Belgique|         50.4|           1|        1|
|        7|Garcia|    Espagne|         18.0|           1|        1|
|        2|Martin|     France|        672.0|           1|        1|
|        1|Durand|     France|        144.6|           1|        2|
|       10|Nguyen|     France|        144.0|           1|        3|
|        6| Rossi|     Italie|         90.0|           1|        1|
|        5| Smith|Royaume-Uni|        276.0|           1|        1|
+---------+------+-----------+-------------+------------+---------+



## Étape 8 – Export

In [35]:
# Étape 8 – Export
ca_par_pays.write.mode("overwrite").parquet("output/ca_par_pays.parquet")
top_clients_par_pays.write.mode("overwrite").parquet("output/top_clients.parquet")
